# Sun hours

Annual peak sun hours per 0.5° grid cell, sourced from NASA POWER climatology (1991–2020, `ALLSKY_SFC_SW_DWN`). Higher = more solar energy at the surface over a year. Ocean cells are masked to `NaN` using `is_land` from `grid.nc`.

NASA POWER publishes surface shortwave irradiance (kWh/m²/day), not literal sunshine-duration hours. We convert to **Peak Sun Hours (PSH)** — the industry-standard translation where annual kWh/m² numerically equals the number of hours of full 1 kW/m² sunshine per year.

In [1]:
import json

import httpx
import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm

from common import RAW_DIR, load_grid, plot_map, save_variable

VARIABLE = 'sun_hours'
variable_raw = RAW_DIR / VARIABLE
variable_raw.mkdir(parents=True, exist_ok=True)

## 1. Fetch raw data

NASA POWER's climatology *regional* endpoint returns per-cell monthly + annual means on a 0.5° lat × 0.625° lon grid. The endpoint caps request area at ~10° × 10°, so we tile the globe into 648 tiles and cache each response as JSON.

First run: ~15–25 min (sequential requests). Re-runs: instant (cached).

In [ ]:
from common import download_nasa_power_dataset
download_nasa_power_dataset(variable_raw, "ALLSKY_SFC_SW_DWN")

## 2. Clean & transform

Each tile JSON is a GeoJSON FeatureCollection. Every feature is a grid point with `geometry.coordinates = [lon, lat]` and `properties.parameter.ALLSKY_SFC_SW_DWN.{JAN..DEC, ANN}` in kWh/m²/day. We take the pre-computed annual mean `ANN` and multiply by 365 to get annual insolation (kWh/m²/year), which equals peak sun hours per year. NASA POWER uses `-999` as its fill value — skip those.

In [ ]:
records = []
for tile_path in sorted(variable_raw.glob('tile_*.json')):
    doc = json.loads(tile_path.read_text())
    for feature in doc['features']:
        lon, lat = feature['geometry']['coordinates'][:2] #ignores elevation in nasa data
        ann = feature['properties']['parameter']['ALLSKY_SFC_SW_DWN']['ANN']
        if ann is None or ann < -900:
            continue
        records.append((lat, lon, ann * 365.0))

df = pd.DataFrame(records, columns=['lat', 'lon', 'sun_hours']).drop_duplicates(['lat', 'lon'])
power = df.set_index(['lat', 'lon'])['sun_hours'].to_xarray().sortby(['lat', 'lon'])
power.attrs['units'] = 'peak sun hours per year (annual kWh/m², NASA POWER ALLSKY_SFC_SW_DWN)'
power

## 3. Interpolate onto the grid

POWER's climatology endpoint returns points on a 1° global grid (cell centres at `.5`). The atlas grid is 0.5° × 0.5°, so bilinear interp upsamples honestly (no invented detail — just smooth transitions between the 1° source points). Then mask ocean cells to NaN.

In [ ]:
grid = load_grid()
values = power.interp(lat=grid.lat, lon=grid.lon, method='linear')
values = values.where(grid.is_land == 1)
values.name = VARIABLE
values

## 4. Plot

In [ ]:
plot_map(values, cmap='YlOrRd')

## 5. Save

In [ ]:
out = save_variable(values, VARIABLE)
print(f'wrote {out}')